In [1]:
import json
import math
import os
import functions_for_experiments
from openai import OpenAI
from sentence_transformers import SentenceTransformer, CrossEncoder
from qdrant_client import QdrantClient, models
from qdrant_client.models import SparseTextEmbedding
from thefuzz import fuzz
from fastembed import SparseTextEmbedding

c:\Users\Olena\Downloads\ucu-employee-support-rag\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 393/393 [00:00<00:00, 4054.99it/s]


In [2]:
with open("final_golden_dataset copy.json", "r", encoding="utf-8") as f:
    golden_data = json.load(f)

In [3]:
n = len(golden_data)

## Baseline

In [4]:
MODEL_E5_BASE = SentenceTransformer('intfloat/multilingual-e5-base')
COLLECTION_E5_BASE = "ucu_documents_e5_base"

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1111.50it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
total_hit_e5_base, total_mrr_e5_base, total_recall_e5_base, total_ndcg_e5_base, \
total_hit_reranked_e5_base, total_mrr_reranked_e5_base, total_recall_reranked_e5_base, total_ndcg_reranked_e5_base = functions_for_experiments.get_metrics(MODEL_E5_BASE, COLLECTION_E5_BASE, e5=True)

In [6]:
print("Results for E5 base model:")
print(f"Hit Rate @ 5: {round(total_hit_e5_base/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_e5_base/n, 2)}")
print(f"Recall @ 5: {round(total_recall_e5_base/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_e5_base/n, 2)}")

Results for E5 base model:
Hit Rate @ 5: 0.64
MRR @ 5: 0.5
Recall @ 5: 0.59
NDCG @ 5: 0.5


In [7]:
print("Results for E5 base model:")
print(f"Hit Rate @ 5: {round(total_hit_reranked_e5_base/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_e5_base/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_e5_base/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_e5_base/n, 2)}")

Results for E5 base model:
Hit Rate @ 5: 0.8
MRR @ 5: 0.73
Recall @ 5: 0.74
NDCG @ 5: 0.7


## HyDE

In [5]:
total_hit_e5_base_hyde, total_mrr_e5_base_hyde, total_recall_e5_base_hyde, total_ndcg_e5_base_hyde, \
total_hit_reranked_e5_base_hyde, total_mrr_reranked_e5_base_hyde, total_recall_reranked_e5_base_hyde, total_ndcg_reranked_e5_base_hyde = functions_for_experiments.get_metrics_hyde(MODEL_E5_BASE, COLLECTION_E5_BASE)

In [6]:
print("Results for E5 base model (HyDE):")
print(f"Hit Rate @ 5: {round(total_hit_e5_base_hyde/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_e5_base_hyde/n, 2)}")
print(f"Recall @ 5: {round(total_recall_e5_base_hyde/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_e5_base_hyde/n, 2)}")

Results for E5 base model (HyDE):
Hit Rate @ 5: 0.34
MRR @ 5: 0.24
Recall @ 5: 0.3
NDCG @ 5: 0.25


In [7]:
print("Results for E5 base model (HyDE):")
print(f"Hit Rate @ 5: {round(total_hit_reranked_e5_base_hyde/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_e5_base_hyde/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_e5_base_hyde/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_e5_base_hyde/n, 2)}")

Results for E5 base model (HyDE):
Hit Rate @ 5: 0.54
MRR @ 5: 0.51
Recall @ 5: 0.46
NDCG @ 5: 0.46


## Query Transform

In [10]:
total_hit_e5_base_transform, total_mrr_e5_base_transform, total_recall_e5_base_transform, total_ndcg_e5_base_transform, \
total_hit_reranked_e5_base_transform, total_mrr_reranked_e5_base_transform, total_recall_reranked_e5_base_transform, total_ndcg_reranked_e5_base_transform = functions_for_experiments.get_metrics_query_transform(MODEL_E5_BASE, COLLECTION_E5_BASE, e5=True)

In [11]:
print("Results for E5 base model (Query Transform):")
print(f"Hit Rate @ 5: {round(total_hit_e5_base_transform/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_e5_base_transform/n, 2)}")
print(f"Recall @ 5: {round(total_recall_e5_base_transform/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_e5_base_transform/n, 2)}")

Results for E5 base model (Query Transform):
Hit Rate @ 5: 0.62
MRR @ 5: 0.41
Recall @ 5: 0.55
NDCG @ 5: 0.41


In [12]:
print("Results for E5 base model (Query Transform):")
print(f"Hit Rate @ 5: {round(total_hit_reranked_e5_base_transform/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_e5_base_transform/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_e5_base_transform/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_e5_base_transform/n, 2)}")

Results for E5 base model (Query Transform):
Hit Rate @ 5: 0.76
MRR @ 5: 0.71
Recall @ 5: 0.72
NDCG @ 5: 0.68


## Hybrid

In [5]:
MODEL_SPARSE = SparseTextEmbedding(model_name="Qdrant/bm25")

In [15]:
total_hit_e5_base_sparse, total_mrr_e5_base_sparse, total_recall_e5_base_sparse, total_ndcg_e5_base_sparse, \
total_hit_reranked_e5_base_sparse, total_mrr_reranked_e5_base_sparse, total_recall_reranked_e5_base_sparse, total_ndcg_reranked_e5_base_sparse = functions_for_experiments.get_metrics_sparse(MODEL_E5_BASE, COLLECTION_E5_BASE, sparse_model=MODEL_SPARSE, e5=True)

In [16]:
print("Results for E5 base model (Hybrid):")
print(f"Hit Rate @ 5: {round(total_hit_e5_base_sparse/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_e5_base_sparse/n, 2)}")
print(f"Recall @ 5: {round(total_recall_e5_base_sparse/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_e5_base_sparse/n, 2)}")

Results for E5 base model (Hybrid):
Hit Rate @ 5: 0.68
MRR @ 5: 0.51
Recall @ 5: 0.61
NDCG @ 5: 0.51


In [17]:
print("Results for E5 base model (Hybrid):")
print(f"Hit Rate @ 5: {round(total_hit_reranked_e5_base_sparse/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_e5_base_sparse/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_e5_base_sparse/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_e5_base_sparse/n, 2)}")

Results for E5 base model (Hybrid):
Hit Rate @ 5: 0.78
MRR @ 5: 0.7
Recall @ 5: 0.71
NDCG @ 5: 0.67


## Query Transform + Hybrid

In [6]:
total_hit_e5_base_transform_hybrid, total_mrr_e5_base_transform_hybrid, total_recall_e5_base_transform_hybrid, total_ndcg_e5_base_transform_hybrid, \
total_hit_reranked_e5_base_transform_hybrid, total_mrr_reranked_e5_base_transform_hybrid, total_recall_reranked_e5_base_transform_hybrid, total_ndcg_reranked_e5_base_transform_hybrid = functions_for_experiments.get_metrics_query_transform(MODEL_E5_BASE, COLLECTION_E5_BASE, e5=True, sparse_model=MODEL_SPARSE)

In [7]:
print("Results for E5 base model (Query Transform + Hybrid):")
print(f"Hit Rate @ 5: {round(total_hit_e5_base_transform_hybrid/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_e5_base_transform_hybrid/n, 2)}")
print(f"Recall @ 5: {round(total_recall_e5_base_transform_hybrid/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_e5_base_transform_hybrid/n, 2)}")

Results for E5 base model (Query Transform + Hybrid):
Hit Rate @ 5: 0.62
MRR @ 5: 0.45
Recall @ 5: 0.55
NDCG @ 5: 0.45


In [8]:
print("Results for E5 base model (Query Transform + Hybrid):")
print(f"Hit Rate @ 5: {round(total_hit_reranked_e5_base_transform_hybrid/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_e5_base_transform_hybrid/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_e5_base_transform_hybrid/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_e5_base_transform_hybrid/n, 2)}")

Results for E5 base model (Query Transform + Hybrid):
Hit Rate @ 5: 0.76
MRR @ 5: 0.69
Recall @ 5: 0.68
NDCG @ 5: 0.65
